In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('CleanedData.csv')
df

In [ ]:
df = df.dropna(axis=1, how='all')
df

In [ ]:
df['time'] = pd.to_datetime(df['time'])

# Add separate columns for year, month, day, day of the week, and hour
df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month
df['day'] = df['time'].dt.day
df['day_of_week'] = df['time'].dt.weekday + 1  # 1 for Monday, 7 for Sunday
df['hour'] = df['time'].dt.hour+1

# Drop the original 'timestamp' column
df = df.drop(columns=['time'])

df

In [ ]:
pd.set_option('display.max_columns', None)  # This will ensure all columns are displayed
print(df.head())  # Show the first 5 rows along with all columns


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Check if using Google Colab
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Device selection
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

names = [
    "hour",
    "price day ahead",
    "day_of_week",
    "day",
    "month",
    "year",
    "temp_Barcelona",
    "pressure_Seville",
    "temp_max_Barcelona",
    "price actual",
    "temp_min_Seville",
    "temp_min_Barcelona",
    "temp_min_Valencia",
    "humidity_Barcelona",
    "pressure_Bilbao",
    "pressure_Barcelona",
    "temp_max_Seville",
    "temp_Madrid",
    "temp_max_Madrid",
    "temp_max_Bilbao",
    "total load actual"
]

# Filter dataset to include only relevant columns
df = df[names]  # Ensure the DataFrame contains only the specified columns

# Splitting dataset
df_train = df[df['year'] < 2018]
df_test = df[df['year'] >= 2018]

# Separate features and target
train_features = df_train.drop(columns=['total load actual']).values
test_features = df_test.drop(columns=['total load actual']).values
train_target = df_train['total load actual'].values
test_target = df_test['total load actual'].values

# Scaling data
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Fit scalers on training data
train_features_scaled = feature_scaler.fit_transform(train_features)
train_target_scaled = target_scaler.fit_transform(train_target.reshape(-1, 1))

# Scale test data
test_features_scaled = feature_scaler.transform(test_features)
test_target_scaled = target_scaler.transform(test_target.reshape(-1, 1))

# Create sequences for model
SEQUENCE_SIZE = 10

def to_sequences(seq_size, features, target):
    x = []
    y = []
    for i in range(len(features) - seq_size):
        window = features[i:(i + seq_size)]
        after_window = target[i + seq_size]
        x.append(window)
        y.append(after_window)
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

x_train, y_train = to_sequences(SEQUENCE_SIZE, train_features_scaled, train_target_scaled)
x_test, y_test = to_sequences(SEQUENCE_SIZE, test_features_scaled, test_target_scaled)

# Data loaders
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

# Transformer Model
class TransformerModel(nn.Module):
    def __init__(self, input_dim=71, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super(TransformerModel, self).__init__()
        self.encoder = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.decoder = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.encoder(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = self.decoder(x[:, -1, :])  # Use the last output for prediction
        return x

# Initialize model, criterion, optimizer, and scheduler
model = TransformerModel(input_dim=len(names) - 1).to(device)  # Adjust input_dim to match the number of features
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3, verbose=True)

# Training Loop
epochs = 100
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        x_batch, y_batch = batch
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in test_loader:
            x_batch, y_batch = batch
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            val_losses.append(loss.item())

    val_loss = np.mean(val_losses)
    scheduler.step(val_loss)

    print(f"Epoch {epoch + 1}/{epochs}, Validation Loss: {val_loss:.4f}")

# Evaluation
model.eval()
predictions_scaled = []
with torch.no_grad():
    for batch in test_loader:
        x_batch, _ = batch
        x_batch = x_batch.to(device)
        outputs = model(x_batch)
        predictions_scaled.extend(outputs.squeeze().tolist())

# Reverse scaling for predictions and actuals
predictions = target_scaler.inverse_transform(np.array(predictions_scaled).reshape(-1, 1)).flatten()
actuals = target_scaler.inverse_transform(y_test.numpy().reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(np.mean((predictions - actuals) ** 2))
print(f"RMSE: {rmse:.2f}")


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Device selection
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

names = [
    "hour",
    "price day ahead",
    "day_of_week",
    "day",
    "month",
    "year",
    "temp_Barcelona",
    "pressure_Seville",
    "temp_max_Barcelona",
    "price actual",
    "temp_min_Seville",
    "temp_min_Barcelona",
    "temp_min_Valencia",
    "humidity_Barcelona",
    "pressure_Bilbao",
    "pressure_Barcelona",
    "temp_max_Seville",
    "temp_Madrid",
    "temp_max_Madrid",
    "temp_max_Bilbao",
    "total load actual"
]

df = df[names]  # Filter dataset

# Splitting dataset
df_train = df[df['year'] < 2018]
df_test = df[df['year'] >= 2018]

# Separate features and target
train_features = df_train.drop(columns=['total load actual']).values
test_features = df_test.drop(columns=['total load actual']).values
train_target = df_train['total load actual'].values
test_target = df_test['total load actual'].values

# Scaling data
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Fit scalers
train_features_scaled = feature_scaler.fit_transform(train_features)
train_target_scaled = target_scaler.fit_transform(train_target.reshape(-1, 1))

# Scale test data
test_features_scaled = feature_scaler.transform(test_features)
test_target_scaled = target_scaler.transform(test_target.reshape(-1, 1))

# Create sequences
SEQUENCE_SIZE = 10

def to_sequences(seq_size, features, target):
    x, y = [], []
    for i in range(len(features) - seq_size):
        x.append(features[i:i + seq_size])
        y.append(target[i + seq_size])
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

x_train, y_train = to_sequences(SEQUENCE_SIZE, train_features_scaled, train_target_scaled)
x_test, y_test = to_sequences(SEQUENCE_SIZE, test_features_scaled, test_target_scaled)

# Data loaders
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# DeepAR Model
class DeepAR(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super(DeepAR, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim, 1)  # Mean
        self.fc_sigma = nn.Linear(hidden_dim, 1)  # Variance

    def forward(self, x):
        output, _ = self.lstm(x)
        hidden_state = output[:, -1, :]
        mu = self.fc_mu(hidden_state)
        sigma = torch.exp(self.fc_sigma(hidden_state))
        return mu, sigma

# Loss Function: Negative Log Likelihood
def nll_loss(mu, sigma, target):
    return torch.mean(0.5 * torch.log(2 * np.pi * sigma**2) + ((target - mu) ** 2) / (2 * sigma**2))

# Training Function
def train_deepar(model, train_loader, test_loader, epochs, optimizer, scheduler, device):
    model.to(device)
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            mu, sigma = model(x_batch)
            loss = nll_loss(mu, sigma, y_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                mu, sigma = model(x_batch)
                loss = nll_loss(mu, sigma, y_batch)
                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)
        scheduler.step(val_loss)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {np.mean(train_losses):.4f}, Val Loss: {val_loss:.4f}")

# Initialize Model
input_dim = x_train.shape[2]
deepar_model = DeepAR(input_dim=input_dim).to(device)
optimizer = torch.optim.Adam(deepar_model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3, verbose=True)

# Train Model
train_deepar(deepar_model, train_loader, test_loader, epochs=100, optimizer=optimizer, scheduler=scheduler, device=device)

# Predictions
deepar_model.eval()
mu_predictions = []
with torch.no_grad():
    for x_batch, _ in test_loader:
        x_batch = x_batch.to(device)
        mu, _ = deepar_model(x_batch)
        mu_predictions.extend(mu.cpu().numpy())

# Reverse scaling
mu_predictions_rescaled = target_scaler.inverse_transform(np.array(mu_predictions).reshape(-1, 1)).flatten()
actuals_rescaled = target_scaler.inverse_transform(y_test.numpy().reshape(-1, 1)).flatten()

# RMSE Calculation
rmse = np.sqrt(np.mean((mu_predictions_rescaled - actuals_rescaled) ** 2))
print(f"DeepAR RMSE: {rmse:.2f}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import statsmodels.api as sm

# Device selection
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

names = [
    "hour",
    "price day ahead",
    "day_of_week",
    "day",
    "month",
    "year",
    "temp_Barcelona",
    "pressure_Seville",
    "temp_max_Barcelona",
    "price actual",
    "temp_min_Seville",
    "temp_min_Barcelona",
    "temp_min_Valencia",
    "humidity_Barcelona",
    "pressure_Bilbao",
    "pressure_Barcelona",
    "temp_max_Seville",
    "temp_Madrid",
    "temp_max_Madrid",
    "temp_max_Bilbao",
    "total load actual"
]

df = df[names]  # Filter dataset

# Splitting dataset
df_train = df[df['year'] < 2018]
df_test = df[df['year'] >= 2018]

# Separate features and target
train_features = df_train.drop(columns=['total load actual']).values
test_features = df_test.drop(columns=['total load actual']).values
train_target = df_train['total load actual'].values
test_target = df_test['total load actual'].values

# Scaling data
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Fit scalers
train_features_scaled = feature_scaler.fit_transform(train_features)
train_target_scaled = target_scaler.fit_transform(train_target.reshape(-1, 1))

# Scale test data
test_features_scaled = feature_scaler.transform(test_features)
test_target_scaled = target_scaler.transform(test_target.reshape(-1, 1))

# Create sequences
SEQUENCE_SIZE = 10

def to_sequences(seq_size, features, target):
    x, y = [], []
    for i in range(len(features) - seq_size):
        x.append(features[i:i + seq_size])
        y.append(target[i + seq_size])
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

x_train, y_train = to_sequences(SEQUENCE_SIZE, train_features_scaled, train_target_scaled)
x_test, y_test = to_sequences(SEQUENCE_SIZE, test_features_scaled, test_target_scaled)

# Data loaders
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# DeepAR Model
class DeepAR(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super(DeepAR, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim, 1)  # Mean
        self.fc_sigma = nn.Linear(hidden_dim, 1)  # Variance

    def forward(self, x):
        output, _ = self.lstm(x)
        hidden_state = output[:, -1, :]
        mu = self.fc_mu(hidden_state)
        sigma = torch.exp(self.fc_sigma(hidden_state))
        return mu, sigma

# Loss Function: Negative Log Likelihood
def nll_loss(mu, sigma, target):
    return torch.mean(0.5 * torch.log(2 * np.pi * sigma**2) + ((target - mu) ** 2) / (2 * sigma**2))

# Training Function
def train_deepar(model, train_loader, test_loader, epochs, optimizer, scheduler, device):
    model.to(device)
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            mu, sigma = model(x_batch)
            loss = nll_loss(mu, sigma, y_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                mu, sigma = model(x_batch)
                loss = nll_loss(mu, sigma, y_batch)
                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)
        scheduler.step(val_loss)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {np.mean(train_losses):.4f}, Val Loss: {val_loss:.4f}")

# Initialize Model
input_dim = x_train.shape[2]
deepar_model = DeepAR(input_dim=input_dim).to(device)
optimizer = torch.optim.Adam(deepar_model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3, verbose=True)

# Train Model
train_deepar(deepar_model, train_loader, test_loader, epochs=100, optimizer=optimizer, scheduler=scheduler, device=device)

# Predictions
deepar_model.eval()
mu_predictions = []
with torch.no_grad():
    for x_batch, _ in test_loader:
        x_batch = x_batch.to(device)
        mu, _ = deepar_model(x_batch)
        mu_predictions.extend(mu.cpu().numpy())

# Reverse scaling
mu_predictions_rescaled = target_scaler.inverse_transform(np.array(mu_predictions).reshape(-1, 1)).flatten()
actuals_rescaled = target_scaler.inverse_transform(y_test.numpy().reshape(-1, 1)).flatten()

# Step 1: Extract Residuals
residuals = actuals_rescaled - mu_predictions_rescaled

# Step 2: Train ARIMA
p, d, q = 1, 1, 1  # ARIMA hyperparameters (tune these as needed)
arima_model = sm.tsa.ARIMA(residuals, order=(p, d, q)).fit()

# Step 3: Forecast Residuals with ARIMA
arima_forecast = arima_model.forecast(steps=len(residuals))
arima_forecast = np.array(arima_forecast)

# Step 4: Combine Predictions
final_predictions = mu_predictions_rescaled + arima_forecast

# Evaluate Combined Model
rmse_combined = np.sqrt(np.mean((final_predictions - actuals_rescaled) ** 2))
print(f"Hybrid DeepAR + ARIMA RMSE: {rmse_combined:.2f}")


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt

# Device selection
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Feature selection
selected_features = [
    "hour",
    "price day ahead",
    "day_of_week",
    "day",
    "month",
    "year",
    "temp_Barcelona",
    "pressure_Seville",
    "temp_max_Barcelona",
    "price actual",
    "temp_min_Seville",
    "temp_min_Barcelona",
    "temp_min_Valencia",
    "humidity_Barcelona",
    "pressure_Bilbao",
    "pressure_Barcelona",
    "temp_max_Seville",
    "temp_Madrid",
    "temp_max_Madrid",
    "temp_max_Bilbao",
    "total load actual"
]

# Filter dataset
df = df[selected_features]

# Train-test split based on the year
train_data = df[df['year'] < 2018]
test_data = df[df['year'] >= 2018]

# Separate features and target for training
X_train = train_data.drop(columns=['total load actual'])
y_train = train_data['total load actual']
X_test = test_data.drop(columns=['total load actual'])
y_test = test_data['total load actual']

# Scaling for DeepAR
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_test_scaled = feature_scaler.transform(X_test)
y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = target_scaler.transform(y_test.values.reshape(-1, 1))

# HistGradientBoostingRegressor
hgb_model = HistGradientBoostingRegressor(random_state=42)
hgb_model.fit(X_train, y_train)

# Predict and calculate RMSE
y_pred_hgb = hgb_model.predict(X_test)
rmse_hgb = np.sqrt(mean_squared_error(y_test, y_pred_hgb))
print(f"HistGradientBoostingRegressor RMSE: {rmse_hgb:.2f}")




In [ ]:
from sklearn.inspection import permutation_importance
import json

# Compute feature importance using permutation importance
perm_importance = permutation_importance(hgb_model, X_test, y_test, n_repeats=10, random_state=42)
feature_importances = perm_importance.importances_mean

# Create the dictionary for model parameters
hgb_model_params = {
    "model_type": "HistGradientBoostingRegressor",
    "params": hgb_model.get_params(),  # Hyperparameters of the model
    "feature_importances": feature_importances.tolist(),  # Feature importance scores as a list
    "rmse": rmse_hgb  # RMSE metric for evaluation
}

# Save to a JSON file
with open("hgb_model.json", "w") as json_file:
    json.dump(hgb_model_params, json_file, indent=4)

print("Model parameters and feature importances saved to 'hgb_model.json'")


In [ ]:
import joblib  # Library to save and load models

# Train the HistGradientBoostingRegressor
hgb_model = HistGradientBoostingRegressor(random_state=42)
hgb_model.fit(X_train, y_train)

# Predict and calculate RMSE
y_pred_hgb = hgb_model.predict(X_test)
rmse_hgb = np.sqrt(mean_squared_error(y_test, y_pred_hgb))
print(f"HistGradientBoostingRegressor RMSE: {rmse_hgb:.2f}")

# Save the trained model as a .pkl file
joblib.dump(hgb_model, "hgb_model.pkl")
print("Model saved as 'hgb_model.pkl'")
# Load the model
loaded_model = joblib.load("hgb_model.pkl")

# Test the loaded model with predictions
test_prediction = loaded_model.predict(X_test)
print(f"Loaded model RMSE: {np.sqrt(mean_squared_error(y_test, test_prediction)):.2f}")
